In [51]:
import pandas as pd
import numpy as np

In [52]:
# =============================================================================
# MERGE SCRIPT — ALL 4 PREPROCESSED TABLES
# =============================================================================
# Join key: year_key = Co_Ref + Renewal_Year (consistent across all tables)
# All interaction tables already filtered to pre-renewal window.
#
# Aggregation strategy:
#   Binary flags          -> max   (1 if ANY call had it)
#   Count                 -> count (total interactions = friction signal)
#   Numeric scores        -> mean, min, last (trend + worst + most recent)
#   High-cardinality cats -> binary explode (preserve ALL signals)
#   Low-cardinality cats  -> meaningful_mode (ignore "Not Mentioned" noise)
# =============================================================================
 
# ── Load ──────────────────────────────────────────────────────────────────────
billings = pd.read_csv(
    "../../dataset/preprocessed/billings_preprocessed.csv"
)
rc = pd.read_csv(
    "../../dataset/preprocessed/renewal_calls_preprocessed.csv"
)
cc = pd.read_csv(
    "../../dataset/preprocessed/cc_calls_preprocessed.csv"
)
emails = pd.read_csv("../../dataset/preprocessed/emails_preprocessed.csv")
 
print(f"Billings      : {billings.shape}")
print(f"Renewal Calls : {rc.shape}")
print(f"CC Calls      : {cc.shape}")
print(f"Emails        : {emails.shape}")

C:\Users\pragi\AppData\Local\Temp\ipykernel_11336\1935459011.py:16: DtypeWarning: Columns (50,51) have mixed types. Specify dtype option on import or set low_memory=False.
  billings = pd.read_csv(


Billings      : (113291, 61)
Renewal Calls : (27412, 38)
CC Calls      : (2123, 34)
Emails        : (22866, 27)


In [53]:
# ── Helpers ───────────────────────────────────────────────────────────────────
# Non-signal values to ignore when finding meaningful mode
NON_SIGNAL = {
    'Not Mentioned', 'Not Discussed', 'Not Applicable',
    'No_Interaction', 'UNAVAILABLE', 'XXXX', '', 'nan'
}
 
def meaningful_mode(s):
    """Mode after filtering out non-signal values."""
    meaningful = s[~s.astype(str).isin(NON_SIGNAL)].dropna()
    if len(meaningful) == 0:
        return 'Not Mentioned'
    return meaningful.mode().iloc[0]
 
def last_value(s):
    """Most recent non-null value (assumes df sorted by Call_Date)."""
    s_clean = s.dropna()
    return s_clean.iloc[-1] if len(s_clean) > 0 else np.nan
 
def binary_explode(df, col, prefix, year_key_col='year_key'):
    """
    For high-cardinality categoricals:
    Create one binary column per unique meaningful value.
    Each column = 1 if ANY call in that cycle had that category.
    e.g. rc_churn_cat_price_value, rc_churn_cat_competitor etc.
    """
    # Filter out non-signal values
    df_clean = df[[year_key_col, col]].copy()
    df_clean = df_clean[~df_clean[col].astype(str).isin(NON_SIGNAL)]
    df_clean = df_clean.dropna(subset=[col])
 
    if len(df_clean) == 0:
        return pd.DataFrame({year_key_col: df[year_key_col].unique()})
 
    # One-hot encode then group by year_key (max = 1 if any call had it)
    dummies = pd.get_dummies(df_clean[col], prefix=prefix)
    dummies[year_key_col] = df_clean[year_key_col].values
    result = dummies.groupby(year_key_col).max().reset_index()
 
    # Clean column names
    result.columns = [
        c.lower().replace(' ', '_').replace('/', '_').replace('-', '_')
        if c != year_key_col else c
        for c in result.columns
    ]
    return result

In [54]:
# =============================================================================
# STEP 1: ADD YEAR KEY TO ALL TABLES
# =============================================================================
billings['year_key'] = (
    billings['Co_Ref'].astype(str) + '_' +
    billings['Renewal_Year'].astype(str)
)
rc['Call_Date'] = pd.to_datetime(rc['Call_Date'])
cc['Call_Date'] = pd.to_datetime(cc['Call_Date'])
rc['year_key'] = (
    rc['Co_Ref'].astype(str) + '_' +
    rc['Call_Date'].dt.year.astype(str)
)
cc['year_key'] = (
    cc['Co_Ref'].astype(str) + '_' +
    cc['Call_Date'].dt.year.astype(str)
)
emails['year_key'] = (
    emails['Co_Ref'].astype(str) + '_' +
    emails['year'].astype(str)
)
 
print(f"\nYear key coverage:")
print(f"  Billings      : {billings['year_key'].nunique()} unique / {len(billings)} rows")
print(f"  Renewal Calls : {rc['year_key'].nunique()} unique cycles / {len(rc)} rows")
print(f"  CC Calls      : {cc['year_key'].nunique()} unique cycles / {len(cc)} rows")
print(f"  Emails        : {emails['year_key'].nunique()} unique cycles / {len(emails)} rows")


Year key coverage:
  Billings      : 113291 unique / 113291 rows
  Renewal Calls : 18579 unique cycles / 27412 rows
  CC Calls      : 1861 unique cycles / 2123 rows
  Emails        : 21362 unique cycles / 22866 rows


In [55]:
# =============================================================================
# STEP 2: RENEWAL CALLS — AGGREGATE
# =============================================================================
print("\nAggregating Renewal Calls...")
 
# Sort by Call_Date so last_value() captures the most recent call
rc = rc.sort_values(['year_key', 'Call_Date'])
 
# Map binary Yes/No -> 1/0
binary_cols_rc = [
    'Serious_Complaint', 'Other_Complaint',
    'Discussion_on_Price_Increase', 'Renewal_Impact_Due_to_Price_Increase',
    'Discount_or_Waiver_Requested', 'Call_Reschedule_Request',
    'Agent_Flagged_Membership_Status_Alert', 'Agent_Renewal_Initiation',
    'Explicit_Competitor_Mention', 'Explicit_Switching_Intent',
    'Mentioned_Competitors', 'Price_Switching_Mentioned',
    'Customer_Asked_For_Justification', 'Discount_Offered',
    'Membership_Renewal_Decision', 'Percentage_Price_Increase_Mentioned',
    'Monetary_Price_Increase_Mentioned',
]
for col in binary_cols_rc:
    if col in rc.columns:
        rc[col] = rc[col].map({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0})
 
# High-cardinality categoricals -> binary explode
# Each unique value becomes its own binary feature
high_card_cats_rc = [
    ('Churn_Category',          'rc_churn_cat'),
    ('Complaint_Category',      'rc_complaint_cat'),
    ('Customer_Reaction_Category', 'rc_reaction_cat'),
    ('Agent_Renewal_Pitch_Category', 'rc_pitch_cat'),
    ('Customer_Renewal_Response_Category', 'rc_response_cat'),
    ('Agent_Response_Category', 'rc_agent_response_cat'),
    ('Desire_To_Cancel',        'rc_desire_cancel'),
    ('Justification_Category',  'rc_justification_cat'),
    ('Reason_For_Renewal_Category', 'rc_renewal_reason_cat'),
    ('Agent_Response_To_Cancel_Category', 'rc_cancel_response_cat'),
    ('Argument_That_Convinced_Customer_to_Stay_Category', 'rc_stay_arg_cat'),
]
 
# Low-cardinality categoricals -> meaningful_mode
low_card_cats_rc = [
    'Call_Direction',
    'Customer_Response',
    'Competitor_Value_Comparison',
]
 
# Base aggregation
agg_rc = {
    'Call_Date': 'count',   # total calls
}
for col in binary_cols_rc:
    if col in rc.columns:
        agg_rc[col] = 'max'
for col in low_card_cats_rc:
    if col in rc.columns:
        agg_rc[col] = meaningful_mode
 
rc_agg = rc.groupby('year_key').agg(agg_rc).reset_index()
rc_agg.rename(columns={'Call_Date': 'rc_total_calls'}, inplace=True)
 
# Prefix low-card cat cols
rc_agg.columns = [
    'year_key' if c == 'year_key'
    else c if c == 'rc_total_calls'
    else f'rc_{c.lower()}' if c in low_card_cats_rc
    else f'rc_{c}'
    for c in rc_agg.columns
]
 
# Merge binary exploded categoricals
for col, prefix in high_card_cats_rc:
    if col in rc.columns:
        exploded = binary_explode(rc, col, prefix)
        rc_agg = rc_agg.merge(exploded, on='year_key', how='left')
 
print(f"Renewal Calls aggregated : {rc_agg.shape}")


Aggregating Renewal Calls...
Renewal Calls aggregated : (18579, 212)


In [56]:
rc_agg.head()

,year_key,rc_total_calls,rc_Serious_Complaint,rc_Other_Complaint,rc_Discussion_on_Price_Increase,rc_Renewal_Impact_Due_to_Price_Increase,rc_Discount_or_Waiver_Requested,rc_Call_Reschedule_Request,rc_Agent_Flagged_Membership_Status_Alert,rc_Agent_Renewal_Initiation,...,rc_stay_arg_cat_financial_incentives___cost_effectiveness,rc_stay_arg_cat_incentive___support_offered,rc_stay_arg_cat_non_renewal___cancellation,rc_stay_arg_cat_other_(misidentified),rc_stay_arg_cat_payment_transparency___flexibility,rc_stay_arg_cat_payment_convenience,rc_stay_arg_cat_payment_terms___renewal_conditions,rc_stay_arg_cat_strong_client_relationship,rc_stay_arg_cat_value_proposition___benefits,rc_stay_arg_cat_value_of_product
0,AA0784_2026,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA0794_2025,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AA0882_2025,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA1246_2024,2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA1394_2024,2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
# =============================================================================
# STEP 3: CC CALLS — AGGREGATE
# =============================================================================
print("Aggregating CC Calls...")

# 1. Sort by Call_Date to ensure 'last_value' is chronologically the latest
cc = cc.sort_values(['year_key', 'Call_Date'])

# 2. Map Binary Columns
binary_cols_cc = [
    'cc_care_package_discussed', 'cc_urgency_getting_on_site',
    'cc_external_consultant', 'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues',
    'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support', 'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact', 'cc_refund_discussed',
    'cc_contractor_suggest_leave', 'cc_contractor_complained',
]

mapping_dict = {True: 1, False: 0, 'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}
for col in binary_cols_cc:
    if col in cc.columns:
        cc[col] = cc[col].map(mapping_dict)

# 3. Numeric Conversion
numeric_cols_cc = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score',
    'days_to_renewal',
]
for col in numeric_cols_cc:
    if col in cc.columns:
        cc[col] = pd.to_numeric(cc[col], errors='coerce')

# 4. Define Aggregation Map
# Use a copy to avoid dictionary mutation issues
agg_cc = {'Call_Date': 'count'}

for col in binary_cols_cc:
    if col in cc.columns:
        agg_cc[col] = 'max'

for col in numeric_cols_cc:
    if col in cc.columns:
        # Note: using a list creates a MultiIndex header
        agg_cc[col] = ['mean', 'min', last_value]

low_card_cats_cc = ['Direction', 'cc_call_initiated_by']
for col in low_card_cats_cc:
    if col in cc.columns:
        # Assuming meaningful_mode is defined elsewhere in your script
        agg_cc[col] = meaningful_mode

# 5. Perform Aggregation
# We keep year_key in the index during flattening to protect it
cc_agg = cc.groupby('year_key').agg(agg_cc)

# 6. Flatten MultiIndex Columns
# This logic checks if a column has sub-levels (like 'mean', 'min') and joins them
new_cols = []
for col_name, agg_func in cc_agg.columns:
    if agg_func == '' or pd.isna(agg_func) or agg_func == 'meaningful_mode':
        new_cols.append(col_name)
    else:
        # Convert function objects like <function last_value> to a string
        func_str = agg_func.__name__ if hasattr(agg_func, '__name__') else str(agg_func)
        new_cols.append(f"{col_name}_{func_str}")

cc_agg.columns = new_cols

# 7. Clean up column names and bring year_key back
cc_agg = cc_agg.reset_index() # year_key is now safely a column
cc_agg.columns = [c.replace('<lambda>', 'last').replace('last_value', 'last') for c in cc_agg.columns]
cc_agg.rename(columns={'Call_Date_count': 'cc_total_calls'}, inplace=True)

# 8. Merge High-Cardinality "Exploded" Columns
high_card_cats_cc = [
    ('cc_contractor_sentiment', 'cc_sentiment'),
    ('cc_care_package',         'cc_package'),
]

for col, prefix in high_card_cats_cc:
    if col in cc.columns:
        exploded = binary_explode(cc, col, prefix)
        # Verify year_key exists in both before merging
        if 'year_key' in exploded.columns and 'year_key' in cc_agg.columns:
            cc_agg = cc_agg.merge(exploded, on='year_key', how='left')
        else:
            print(f"Warning: Could not merge {col}. year_key missing.")

print(f"CC Calls aggregated      : {cc_agg.shape}")

Aggregating CC Calls...
CC Calls aggregated      : (1861, 45)


In [58]:
cc_agg.head()

,year_key,cc_total_calls,cc_care_package_discussed_max,cc_urgency_getting_on_site_max,cc_external_consultant_max,cc_agent_cross_sell_attempt_max,cc_customer_issues_concerns_max,cc_business_struggles_financial_hardship_max,cc_questionnaire_completion_max,cc_chasing_response_max,...,days_to_renewal_min,days_to_renewal_last,Direction,cc_call_initiated_by,cc_sentiment_dissatisfied,cc_sentiment_neutral,cc_sentiment_satisfied,cc_package_express,cc_package_premier,cc_package_standard
0,AA0994_2025,7,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1,...,2.0,2.0,IN_BOUND,Customer,True,False,True,True,False,False
1,AD4863_2025,1,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0,...,3.0,3.0,OUT_BOUND,Agent,False,False,True,True,False,False
2,AD7382_2025,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,...,4.0,4.0,OUT_BOUND,Agent,False,False,True,NaN,NaN,NaN
3,AD7903_2025,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,...,12.0,12.0,OUT_BOUND,Customer,False,True,False,NaN,NaN,NaN
4,AE5853_2025,1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,...,13.0,13.0,IN_BOUND,Customer,False,False,True,NaN,NaN,NaN


In [59]:
# =============================================================================
# STEP 4: EMAILS — AGGREGATE
# =============================================================================
print("Aggregating Emails...")
 
# Binary cols -> 1/0
binary_cols_em = [
    'crm_accreditation_completed', 'crm_timely_completion',
    'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_agent_chased_contractor', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_dissatisified_with_renewal_price',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]
for col in binary_cols_em:
    if col in emails.columns:
        emails[col] = emails[col].map(
            {'Yes': 1, 'No': 0, 'Not Discussed': 0,
             'yes': 1, 'no': 0}
        )
 
# Numeric cols
numeric_cols_em = ['crm_contractor_sentiment_score', 'crm_agent_chase_count']
for col in numeric_cols_em:
    if col in emails.columns:
        emails[col] = pd.to_numeric(emails[col], errors='coerce')
 
# High-cardinality cats -> binary explode
high_card_cats_em = [
    ('crm_contractor_sentiment', 'em_sentiment'),
    ('crm_membership_level',     'em_membership_level'),
]
 
# Low-cardinality cats -> meaningful_mode
low_card_cats_em = ['crm_auto_renewal_status']
 
agg_em = {'Time_to_Renewal': 'count'}
for col in binary_cols_em:
    if col in emails.columns:
        agg_em[col] = 'max'
for col in numeric_cols_em:
    if col in emails.columns:
        agg_em[col] = 'mean'
for col in low_card_cats_em:
    if col in emails.columns:
        agg_em[col] = meaningful_mode
 
em_agg = emails.groupby('year_key').agg(agg_em).reset_index()
em_agg.rename(columns={'Time_to_Renewal': 'em_total_emails'}, inplace=True)
 
# Merge binary exploded categoricals
for col, prefix in high_card_cats_em:
    if col in emails.columns:
        exploded = binary_explode(emails, col, prefix)
        em_agg = em_agg.merge(exploded, on='year_key', how='left')
 
print(f"Emails aggregated        : {em_agg.shape}")

Aggregating Emails...
Emails aggregated        : (21362, 31)


In [60]:
em_agg.head()

,year_key,em_total_emails,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_dts_or_ssip_mentioned,crm_customer_payment_intention,...,crm_contractor_sentiment_score,crm_agent_chase_count,crm_auto_renewal_status,em_sentiment_dissatisfied,em_sentiment_neutral,em_sentiment_satisfied,em_membership_level_accredited,em_membership_level_in_progress,em_membership_level_premier,em_membership_level_standard
0,AA0784_2026,1,0,0,0,0,0,0,0,0,...,50.0,1.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA0794_2025,1,0,0,0,0,0,0,1,0,...,50.0,2.0,0,NaN,NaN,NaN,True,False,False,False
2,AA0882_2025,1,0,0,0,0,0,1,0,1,...,50.0,1.0,0,False,True,False,NaN,NaN,NaN,NaN
3,AA0923_2026,1,0,0,0,0,0,0,0,0,...,50.0,1.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA0994_2025,1,1,0,0,0,1,1,0,0,...,50.0,0.0,2,False,True,False,True,False,False,False


In [61]:
# =============================================================================
# STEP 5: FINAL LEFT JOINS ONTO BILLINGS
# =============================================================================
print("\nMerging onto Billings...")
 
df = billings.merge(rc_agg, on='year_key', how='left')
print(f"  After RC merge  : {df.shape}")
 
df = df.merge(cc_agg, on='year_key', how='left')
print(f"  After CC merge  : {df.shape}")
 
df = df.merge(em_agg, on='year_key', how='left')
print(f"  After EM merge  : {df.shape}")


Merging onto Billings...
  After RC merge  : (113291, 273)
  After CC merge  : (113291, 317)
  After EM merge  : (113291, 347)


In [62]:
# =============================================================================
# STEP 6: POST-MERGE NULL FILL
# =============================================================================
 
# Count cols -> 0
for col in ['rc_total_calls', 'cc_total_calls', 'em_total_emails']:
    if col in df.columns:
        df[col] = df[col].fillna(0)
 
# Binary flag cols -> 0 (no interaction = no signal)
for col in df.columns:
    if df[col].dtype in ['float64', 'int64', 'Int64']:
        if (col.startswith('rc_') or col.startswith('cc_')
                or col.startswith('crm_') or col.startswith('em_')):
            df[col] = df[col].fillna(0)
 
# Categorical cols -> 'No_Interaction'
for col in df.columns:
    if df[col].dtype == object:
        if (col.startswith('rc_') or col.startswith('cc_')
                or col.startswith('crm_') or col.startswith('em_')):
            df[col] = df[col].fillna('No_Interaction')

In [63]:
df.info


<bound method DataFrame.info of         Co_Ref Renewal_Month  Connection_Net  Connection_Qty Discount_Amount  \
0       VT6174    2024-11-01             0.0             0.0               0   
1       VD3828    2025-08-01             0.0             0.0               0   
2       DV8120    2025-03-01             0.0             0.0               0   
3       EZ9894    2025-06-01             0.0             0.0               0   
4       FA8957    2025-03-01             0.0             0.0               0   
...        ...           ...             ...             ...             ...   
113286  VI6211    2023-04-01             0.0             0.0               0   
113287  NI7208    2023-06-01             0.0             0.0               0   
113288  DO6023    2023-09-01             0.0             0.0               0   
113289  KR5815    2023-08-01             0.0             0.0               0   
113290  MA8737    2023-07-01             0.0             0.0               0   

       

In [64]:
high_null_cols = [
    "days_to_renewal_mean",
    "days_to_renewal_min",
    "days_to_renewal_last",
    "Direction"
]

df = df.drop(columns=high_null_cols)

In [72]:
# Dates
df["Registration_Date"] = pd.to_datetime(df["Registration_Date"], errors="coerce")
df["Prospect_Renewal_Date"] = pd.to_datetime(df["Prospect_Renewal_Date"], errors="coerce")
df["Proforma_Date"] = pd.to_datetime(df["Proforma_Date"], errors="coerce")

# 1. History → missing = new customer
df["Last_Band"].fillna("New_Customer", inplace=True)
df["Last_Renewal"].fillna("No_History", inplace=True)

# 2. Payment → missing = 0 (not paid)
df["Total_Net_Paid"].fillna(0, inplace=True)

# 3. Proforma → missing = not started
df["Proforma_Account_Stage"].fillna("Not_Started", inplace=True)
df["Proforma_Audit_Status"].fillna("Unknown", inplace=True)

# 4. Tenure → missing = 0 / new
df["Tenure_Years"].fillna(0, inplace=True)
df["Tenure_Group"].fillna("New", inplace=True)

# 5. Proforma delay (derived feature)
df["proforma_delay"] = (df["Prospect_Renewal_Date"] - df["Proforma_Date"]).dt.days
df["proforma_delay"].fillna(-1, inplace=True)  # -1 = no proforma

# 6. Small missing → simple defaults
df["Anchor_Group"].fillna("Unknown", inplace=True)
df["Renewal_Score_At_Release"].fillna(df["Renewal_Score_At_Release"].median(), inplace=True)
df["Proforma_Membership_Status"].fillna("Unknown", inplace=True)
df["Proforma_Approved_Lists"].fillna(0, inplace=True)
df["Connection_Group"].fillna("Unknown", inplace=True)
df["#_of_Connection"].fillna(0, inplace=True)

# 7. Very small missing
df["Band"].fillna(df["Band"].mode()[0], inplace=True)


df["missing_proforma"] = df["Proforma_Date"].isna().astype(int)

# Drop rows where either Registration_Date OR Proforma_Date is missing
df = df.dropna(subset=['Registration_Date', 'Proforma_Date'])

C:\Users\pragi\AppData\Local\Temp\ipykernel_11336\4121224802.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Last_Band"].fillna("New_Customer", inplace=True)
C:\Users\pragi\AppData\Local\Temp\ipykernel_11336\4121224802.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For

In [73]:
# =============================================================================
# STEP 7: FINAL CHECKS
# =============================================================================
print("\n" + "="*60)
print("FINAL MERGED TABLE")
print("="*60)
print(f"Shape      : {df.shape}")
print(f"Churn rate : {df['churn_label'].mean():.2%}")
 
print(f"\nInteraction coverage:")
print(f"  Has renewal calls : {(df['rc_total_calls'] > 0).sum():>6} rows  ({(df['rc_total_calls'] > 0).mean():.1%})")
print(f"  Has CC calls      : {(df['cc_total_calls'] > 0).sum():>6} rows  ({(df['cc_total_calls'] > 0).mean():.1%})")
print(f"  Has emails        : {(df['em_total_emails'] > 0).sum():>6} rows  ({(df['em_total_emails'] > 0).mean():.1%})")
 
print(f"\nRemaining nulls (top 10):")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(10))
 
# Save
df.to_csv("../../dataset/preprocessed/final_merged.csv", index=False)
print(f"\nSaved : dataset/preprocessed/final_merged.csv")
print(f"Ready for EDA and feature engineering.")


FINAL MERGED TABLE
Shape      : (112089, 345)
Churn rate : 10.71%

Interaction coverage:
  Has renewal calls :  18380 rows  (16.4%)
  Has CC calls      :   1851 rows  (1.7%)
  Has emails        :  19966 rows  (17.8%)

Remaining nulls (top 10):
Series([], dtype: int64)



Saved : dataset/preprocessed/final_merged.csv
Ready for EDA and feature engineering.


In [74]:
df = pd.read_csv("../../dataset/preprocessed/final_merged.csv")

C:\Users\pragi\AppData\Local\Temp\ipykernel_11336\1728635200.py:1: DtypeWarning: Columns (4,47,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../dataset/preprocessed/final_merged.csv")


In [75]:
from pathlib import Path
 
out_path = Path('column_uniques.txt')
 
# detect id-like columns to exclude: name contains id/ref/code or every value is unique
n_rows = len(df)
id_like = []
for col in df.columns:
    lname = col.lower()
    if ('id' in lname.split() or 'id' in lname or 'Co_Ref' in lname or 'code' in lname) and (df[col].nunique(dropna=False) > 1):
        # name contains id/ref/code -> mark as id-like
        id_like.append(col)
    elif df[col].nunique(dropna=False) == n_rows:
        # all values unique -> likely an identifier
        id_like.append(col)
 
# make unique list
id_like = sorted(set(id_like))
 
def simple_dtype(col_series):
    if pd.api.types.is_numeric_dtype(col_series):
        return 'numerical'
    if pd.api.types.is_datetime64_any_dtype(col_series):
        return 'datetime'
    return 'categorical'
 
with out_path.open('w', encoding='utf-8') as f:
    f.write(f"Detected id-like columns (excluded): {', '.join(id_like) if id_like else 'None'}\n")
    f.write(f"Rows: {n_rows}\n\n")
    for col in df.columns:
        if col in id_like:
            continue
        ser = df[col]
        dtype_label = simple_dtype(ser)
        total_unique = ser.nunique(dropna=False)
        missing_count = ser.isnull().sum()
        missing_pct = (missing_count / n_rows) * 100 if n_rows else 0.0
 
        f.write(f"Column: {col}\n")
        f.write(f"Type: {dtype_label}\n")
        f.write(f"Unique count (including NaN): {total_unique}\n")
        f.write(f"Missing: {missing_count} ({missing_pct:.2f}%)\n")
 
        # Prepare values: show all if small, otherwise show sample + count
        non_null_unique = ser.dropna().unique()
        nunique_non_null = len(non_null_unique)
        if nunique_non_null == 0:
            f.write("Values: [all values are NaN]\n")
        elif nunique_non_null <= 50:
            vals = ', '.join(map(str, list(non_null_unique)))
            f.write(f"Values ({nunique_non_null}): {vals}\n")
        else:
            sample_vals = ', '.join(map(str, list(non_null_unique[:50])))
            f.write(f"Values: {nunique_non_null} unique values (showing first 50): {sample_vals}\n")
 
        f.write("-" * 80 + "\n")
 
print(f"Wrote column unique summary to: {out_path.resolve()}")

Wrote column unique summary to: C:\Projects\06_Pre-renewal_churn\notebooks\01_preprocessing\column_uniques.txt
